# Day 1 · 01 · Intro to Lakehouse & Delta Lake

## Business Scenario

RetailHub has a familiar Power BI problem: analysts can connect to many tables,
but they do not yet know which compute endpoint, catalog object or table version
should be trusted for reporting. Day 1 starts by turning Databricks from "another
SQL endpoint" into a governed BI platform: SQL Warehouse for compute, Unity
Catalog for discovery and Delta Lake for reliable tables.


## Learning Objectives

By the end of this notebook participants can:

- explain the Lakehouse model in BI terms,
- identify the role of Unity Catalog, SQL Warehouse, SQL Editor and Catalog Explorer,
- inspect Delta metadata and history without treating Delta as a black box,
- connect warehouse sizing and auto-stop decisions to BI cost,
- discover source grain before building a semantic model,
- explain time travel, `OPTIMIZE`, `VACUUM` and constraints at an analyst level.


In [ ]:
%run ../../setup/00_setup

### Configuration

Display the active catalog context for this session.

In [ ]:
display(spark.createDataFrame([
    ("CATALOG", CATALOG),
    ("BRONZE",  BRONZE),
    ("SILVER",  SILVER),
    ("GOLD",    GOLD),
], ["Variable", "Value"]))

### Runtime Pre-check

Fail fast if the required Silver tables are missing. Run `data/generate_training_dataset.ipynb` first if this cell raises an exception.

In [ ]:
required_tables = [
    f"{SILVER}.customers",
    f"{SILVER}.products",
    f"{SILVER}.sales_orders",
    f"{SILVER}.order_lines",
]
missing = [t for t in required_tables if not spark.catalog.tableExists(t)]

In [ ]:
if missing:
    for t in missing: print("[MISSING]", t)
    raise Exception("Pre-check failed. Run data/generate_training_dataset.ipynb first.")
print("[OK] All required tables exist:")
for t in required_tables: print(" -", t)

## 1. What is the Lakehouse?

The Lakehouse combines the openness of a data lake with the governance and performance of a data warehouse. Instead of maintaining two separate systems, all layers — raw, cleansed, and curated — live in the same open storage.

| | OLTP (transactional) | Classic DWH | Databricks Lakehouse |
|---|---|---|---|
| Storage | Row-based DB | Proprietary format | Open Parquet files |
| Compute | Same system | Proprietary engine | Decoupled (SQL Warehouse) |
| Governance | DB permissions | Siloed | Unity Catalog (row/col level) |
| BI contract | No | ETL → marts | Gold layer (versioned, testable) |
| Scale cost | Expensive | Expensive | Pay-per-query |

### Lakehouse decision map for Power BI teams

| Question from BI | Lakehouse answer | Why it matters |
|---|---|---|
| Where is the approved table? | Unity Catalog object | avoids connecting to raw or personal tables |
| Which compute should Power BI use? | SQL Warehouse | isolates BI workload from notebooks/jobs |
| How do we know a refresh was complete? | Delta transaction log | reports do not read half-written data |
| Where do business definitions live? | Gold layer comments, views and tables | reduces duplicated Power Query/DAX logic |
| How do we audit a change? | table history and lineage | explains why yesterday's KPI changed |

The recurring rule for the whole course: Power BI should consume curated Gold
objects, not rediscover business logic from Silver every time a report is built.


![Medallion to Power BI](../../assets/images/medallion_to_powerbi.png)

Unity Catalog is the control plane — catalog, schemas, tables, lineage, and access control in one place.

List all catalogs visible to the current user. In a well-governed workspace each team or environment has its own catalog.

In [ ]:
%sql
SHOW CATALOGS

Each catalog contains schemas (databases). The training catalog uses `bronze`, `silver`, and `gold` to represent the medallion layers.

In [ ]:
display(spark.sql(f"SHOW SCHEMAS IN {CATALOG}"))

## 2. Delta Tables & Delta Log

A Delta table = Parquet data files + a JSON transaction log (`_delta_log/`). Every write appends a new log entry. This is what enables time travel, ACID transactions, and schema enforcement.

`DESCRIBE DETAIL` shows the physical layout: format, number of files, total size, and storage location.

In [ ]:
%sql
DESCRIBE DETAIL silver.customers

The `location` field shows where files live in object storage. Let’s look at the raw `_delta_log/` directory to see the transaction files.

In [ ]:
detail   = spark.sql("DESCRIBE DETAIL silver.customers").collect()[0]
log_path = detail["location"] + "/_delta_log/"
display(dbutils.fs.ls(log_path))

Each `.json` file in `_delta_log/` is one transaction (add, remove, schema change). Databricks reads them in sequence to reconstruct the current table state.

`DESCRIBE HISTORY` lists every transaction with its version number, timestamp, operation, and metrics — the audit trail of the table.

In [ ]:
%sql
DESCRIBE HISTORY silver.customers LIMIT 5

`DESCRIBE EXTENDED` adds column-level metadata, table properties, and catalog registration details on top of the basic schema.

In [ ]:
%sql
DESCRIBE EXTENDED silver.order_lines

### Delta capabilities that protect BI reports

| Delta capability | Analyst-level explanation | BI impact |
|---|---|---|
| ACID transactions | each write is committed as one table version | dashboards avoid partial refreshes |
| schema enforcement | incompatible writes fail | bad columns do not silently corrupt reports |
| schema evolution | allowed changes are explicit | semantic model changes can be planned |
| constraints | stable data rules can be enforced at table level | invalid facts fail before reporting |
| time travel | query a previous table version | reproduce yesterday's KPI during debugging |
| transaction history | inspect operations and metrics | explain refreshes, deletes and overwrites |

For this analyst course, the goal is not to administer Delta. The goal is to
recognize which Delta features make Gold trustworthy enough for Power BI.


### Disposable Delta contract demo

This creates a tiny demo table, adds a constraint and a planned schema change,
then reads its history. It is intentionally dropped at the end so the demo does
not leave extra Gold objects behind.


In [ ]:
%sql
DROP TABLE IF EXISTS gold.delta_contract_demo


In [ ]:
%sql
CREATE TABLE gold.delta_contract_demo
USING DELTA
AS
SELECT line_id, order_id, product_id, order_date, quantity
FROM silver.order_lines
WHERE quantity > 0
LIMIT 1000


In [ ]:
%sql
ALTER TABLE gold.delta_contract_demo
ADD CONSTRAINT positive_quantity CHECK (quantity > 0)


In [ ]:
%sql
ALTER TABLE gold.delta_contract_demo
ADD COLUMNS (contract_note STRING)


In [ ]:
%sql
DESCRIBE HISTORY gold.delta_contract_demo LIMIT 10


In [ ]:
%sql
DROP TABLE IF EXISTS gold.delta_contract_demo


## 3. SQL Warehouse — What It Is & Pricing

A SQL Warehouse is a compute endpoint — like a connection string in SQL Server, but with its own isolated compute pool. Storage is separate (your Delta files in object storage). The warehouse just runs queries.

![SQL Warehouse cost decision](../../assets/images/sql_warehouse_cost_decision.png)

Choose the warehouse type based on your workload pattern:

| | Serverless | Pro | Classic |
|---|---|---|---|
| Billing | Per-second | Per-minute | Per-minute |
| Cold start | ~2–5 seconds | ~20–40 seconds | ~2–3 min |
| Best for | Ad-hoc queries, BI dashboards | Scheduled workloads | Legacy / specific VNet requirements |
| Auto-stop | Yes (configurable) | Yes | Yes |
| Photon | Yes | Yes | No (by default) |

```
[UI DEMO] SQL Warehouses walkthrough:
1. Left sidebar → SQL Warehouses
2. Click warehouse name → Connection details tab
   → copy Server hostname + HTTP path (needed for Power BI)
3. Configuration tab → Auto-stop setting
4. Monitoring tab → Query history + active queries
```

### Warehouse cost guardrails for BI

| Guardrail | Good starting point | Trainer discussion |
|---|---|---|
| Warehouse type | Serverless for interactive BI | fastest start and easiest operations if available |
| Size | start small, scale by evidence | use Query History before increasing size |
| Auto-stop | 5-10 minutes for training/dev | prevents idle DBU spend between exercises |
| Workload split | separate BI from ETL jobs | prevents dashboards and pipelines competing |
| Query ownership | named users/service principals | makes cost and slow query follow-up possible |
| Expensive visuals | aggregate or Import mode | do not make every slicer scan line-level facts |

Cost governance is not a separate finance topic. It is a modeling topic: a bad
Gold design becomes expensive when Power BI repeats the same inefficient query
for every visual and every user.


Confirm the active user, catalog context, and which SQL Warehouse this session is running on.

In [ ]:
%sql
SELECT
  current_user()       AS current_user,
  current_catalog()    AS catalog,
  current_schema()     AS schema,
  current_warehouse()  AS warehouse

`system.billing.usage` shows DBU consumption per SKU over the last 30 days. This is account-level and may not be available in every workspace.

In [ ]:
try:
    sql = "SELECT cloud, sku_name, SUM(usage_quantity) AS dbu_usage FROM system.billing.usage WHERE usage_start_time >= current_date() - 30 GROUP BY 1, 2 ORDER BY 3 DESC LIMIT 5"
    display(spark.sql(sql))
except Exception:
    print("[INFO] system.billing.usage not available in this workspace.")
    print("       Trainer can show billing in Account Console -> Usage.")

## 4. SQL Editor & Catalog Explorer

SQL Editor and notebooks both run SQL against the same warehouse. The choice depends on the workflow:

| | SQL Editor | Notebook |
|---|---|---|
| Best for | Ad-hoc queries, quick exploration | Multi-step pipelines, documentation |
| Output | Inline table, chart, download | Inline + Python objects + charts |
| Version control | Query history only | Git-backed (.ipynb) |
| Sharing | Link to query | Share notebook |
| BI export | Direct to Dashboard | Via display() → chart |

![SQL Editor](../../assets/images/source_sql_editor.png)

```
[UI DEMO] Catalog Explorer:
1. Left sidebar → Catalog
2. Expand: retailhub_[your_name] → silver → customers
3. Click table name → see: columns, sample data, comments
4. Click Lineage tab → see where data comes from / goes to
5. For Power BI developers: Catalog Explorer is your schema documentation
```

![Catalog Explorer lineage](../../assets/images/source_catalog_explorer_lineage.png)

Three things Power BI developers look for in Catalog Explorer: (1) column names and types — matches what you’ll see in Power Query, (2) table comments — business definitions, (3) lineage — understand if Gold is downstream of Silver.

## 5. Source Data Discovery

### Discovery checklist before any Gold model

Before creating dimensions or facts, answer these questions explicitly:

| Question | Example in RetailHub | Common BI failure if skipped |
|---|---|---|
| What is the grain? | order header vs order line | orders counted as lines |
| Which status counts as revenue? | `Completed` only | revenue overstated by returns/cancellations |
| Which keys join dimensions? | customer_id, product_id | orphan rows become blank slicers |
| Which dates drive reporting? | order_date | wrong fiscal/month trend |
| Which fields are measures? | quantity, revenue, margin | accidental aggregation of attributes |
| Which table is safe for BI? | Gold, not Silver | duplicated logic in every PBIX |


Open `sql/exploration_queries.sql` in SQL Editor (navigate to it in the repo file browser). Run each query there. Then run the same queries below — the results should match exactly.

Row counts for all four Silver tables. A mismatch between runs indicates an incomplete data load.

In [ ]:
%sql
SELECT 'customers'    AS tbl, COUNT(*) AS rows FROM silver.customers
UNION ALL SELECT 'products',    COUNT(*) FROM silver.products
UNION ALL SELECT 'sales_orders', COUNT(*) FROM silver.sales_orders
UNION ALL SELECT 'order_lines',  COUNT(*) FROM silver.order_lines

Date range and grain of the orders table. `COUNT(*) = COUNT(DISTINCT order_id)` confirms one row per order.

In [ ]:
%sql
SELECT
  MIN(order_date)             AS min_date,
  MAX(order_date)             AS max_date,
  COUNT(*)                    AS total_orders,
  COUNT(DISTINCT order_id)    AS distinct_orders
FROM silver.sales_orders

Status distribution reveals which business states exist in the data. KPI definitions must agree on which statuses count as revenue.

In [ ]:
%sql
SELECT status, COUNT(*) AS cnt
FROM silver.sales_orders
GROUP BY status
ORDER BY cnt DESC

Grain check on order lines: average lines per order confirms this is a detail table, not a header table.

In [ ]:
%sql
SELECT
  COUNT(*)                                      AS total_lines,
  COUNT(DISTINCT order_id)                      AS distinct_orders,
  ROUND(COUNT(*) / COUNT(DISTINCT order_id), 2) AS avg_lines_per_order
FROM silver.order_lines

Grain confirmed: one row per order line. Multiple lines per order (avg shown above). This grain carries forward to Gold.

## 6. Delta Lake Deep Dive — Time Travel & RESTORE

Time travel queries the table at a past version. Use it for: (1) auditing what data looked like before a load, (2) debugging incorrect aggregations, (3) reproducing a historical report. Do NOT use it as a backup strategy.

Check the available history versions before querying a past state.

In [ ]:
%sql
DESCRIBE HISTORY silver.customers LIMIT 3

`VERSION AS OF 0` reads the table exactly as it was after the first write — before any updates or deletes.

In [ ]:
%sql
SELECT COUNT(*) AS rows_at_version_0
FROM silver.customers VERSION AS OF 0

RESTORE TABLE rolls the table back to a past version. Use it only after an accidental DELETE or incorrect UPDATE — and only on a table you own.

```sql
-- RESTORE syntax (do not run on training data):
-- RESTORE TABLE silver.customers TO VERSION AS OF 3;
-- RESTORE TABLE silver.customers TO TIMESTAMP AS OF '2025-01-01 00:00:00';
```

OPTIMIZE and VACUUM are maintenance commands run by the data engineer, not the analyst: OPTIMIZE rewrites small files into larger ones (faster queries), VACUUM deletes old data files that are no longer needed (storage savings). Default retention: 7 days.

Table properties show the current Delta configuration, including retention settings that control how far back time travel is available.

In [ ]:
%sql
DESCRIBE DETAIL silver.order_lines